# Scientific diagnostics — final Ridge25 downscaling

Diagnostic notebook for the frozen five-year production workflow.

**Frozen scientific source**
- run: `20260907T162048Z_2020_2024`
- source commit: `d0ced516d24f42f724081c63c9ab3c8f4c2687a8`
- target: `Kc_target = MODIS ET / ETo`
- final model: Ridge25
- spatial domain control: equal-weight standardized DI/AOA
- fine-resolution production: exact-overlap conservative reconciliation + support90

The design follows the descriptive logic of the former 30 m diagnostic,
but it does **not** reproduce methodological elements that are no longer part
of the accepted workflow.

Important terminology:
- `Kc_target` is a **MODIS-derived training target**, not an in-situ observation.
- Field ET is a **field-derived ET proxy**, not independent validation of a 20 m pixel.
- Standardized Ridge coefficients describe the fitted linear structure; under
  collinearity they must not be interpreted as causal variable importance.
- Conservation is assessed on the full reconciled MODIS support **before**
  the final publication mask.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from scipy.stats import spearmanr

FINAL_RUN_ID = "20260907T162048Z_2020_2024"
FINAL_SOURCE_COMMIT = "d0ced516d24f42f724081c63c9ab3c8f4c2687a8"
FINAL_DATES = ["2020-03-13", "2021-11-25", "2022-03-30"]

STATION_NAMES = {
    "ST01": "Clean pasture",
    "ST02": "Oil palm",
    "ST03": "Banana",
    "ST04": "Mangrove",
    "ST05": "Dry forest",
}

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "et_downscaling").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Repository root could not be located.")

def require(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def save_figure(fig: plt.Figure, name: str) -> Path:
    path = FIGURE_DIR / name
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")
    return path

def regression_metrics(observed, predicted):
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = predicted - observed

    ss_res = np.sum((observed - predicted) ** 2)
    ss_tot = np.sum((observed - np.mean(observed)) ** 2)
    r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot
    rmse = np.sqrt(np.mean(error ** 2))
    mae = np.mean(np.abs(error))
    bias = np.mean(error)

    if len(observed) > 1 and np.std(observed) > 0 and np.std(predicted) > 0:
        r = np.corrcoef(observed, predicted)[0, 1]
    else:
        r = np.nan

    return {
        "n": len(observed),
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "BIAS": bias,
        "r": r,
    }

def identity_limits(*arrays):
    values = np.concatenate([np.asarray(a, dtype=float) for a in arrays])
    values = values[np.isfinite(values)]
    lower = float(values.min())
    upper = float(values.max())
    margin = 0.05 * max(upper - lower, 1e-6)
    return lower - margin, upper + margin

REPO_ROOT = find_repo_root(Path.cwd().resolve())
WORKSPACE = REPO_ROOT.parent / "ET_fundacion_workspace"
RUN_DIR = require(WORKSPACE / "current" / "runs" / FINAL_RUN_ID)
TABLE_DIR = require(RUN_DIR / "tables")
FIGURE_DIR = WORKSPACE / "current" / "diagnostics" / "scientific_figures_final"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FIELD_DIR = require(
    WORKSPACE
    / "current"
    / "diagnostics"
    / "field_ridge25_final_scenarios"
)

RUN_METADATA = require(RUN_DIR / "run_metadata.json")
TRAINING = require(TABLE_DIR / "ridge25_training_population.csv")
SPATIAL_OOF = require(TABLE_DIR / "ridge25_spatial_oof.csv")
TEMPORAL_OOF = require(TABLE_DIR / "ridge25_loyo_oof.csv")
SPATIAL_FOLDS = require(TABLE_DIR / "ridge25_spatial_fold_metrics.csv")
TEMPORAL_FOLDS = require(TABLE_DIR / "ridge25_loyo_fold_metrics.csv")
STATION_METRICS = require(TABLE_DIR / "ridge25_spatial_metrics_by_station.csv")
MODEL_PARAMETERS = require(TABLE_DIR / "ridge25_model_parameters.csv")
VALIDATION_METRICS = require(TABLE_DIR / "ridge25_validation_metrics.csv")
PERSISTENCE = require(TABLE_DIR / "persistence_baseline_metrics.csv")
AOA_TRAINING = require(TABLE_DIR / "ridge25_aoa_training_di.csv")
AOA_METADATA = require(RUN_DIR / "ridge25_aoa_metadata.json")

FIELD_COMPARISON = require(FIELD_DIR / "field_comparison_scenarios.csv")
FIELD_METRICS = require(FIELD_DIR / "field_scenario_metrics.csv")
FIELD_SENSITIVITY = require(FIELD_DIR / "field_temporal_completeness_sensitivity.csv")
RASTER_SUMMARY = require(WORKSPACE / "final" / "raster_summary.csv")

metadata = json.loads(RUN_METADATA.read_text(encoding="utf-8"))
aoa_metadata = json.loads(AOA_METADATA.read_text(encoding="utf-8"))

assert metadata["provenance"]["repository"]["commit"] == FINAL_SOURCE_COMMIT
assert metadata["provenance"]["repository"]["dirty"] is False

print("Repository:", REPO_ROOT)
print("Workspace:", WORKSPACE)
print("Run:", FINAL_RUN_ID)
print("Source commit:", FINAL_SOURCE_COMMIT)
print("Figure directory:", FIGURE_DIR)

## D01 — Overall validation: spatial transfer versus temporal transfer

This is the direct analogue of the former validation-scheme diagnostic,
but only the two accepted OOF protocols are shown. No random split is used
as evidence of final performance.

In [ ]:
spatial = pd.read_csv(SPATIAL_OOF, parse_dates=["period_start"])
temporal = pd.read_csv(TEMPORAL_OOF, parse_dates=["period_start"])
validation = pd.read_csv(VALIDATION_METRICS)

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.3))

for ax, table, title in [
    (axes[0], spatial, "Spatial block OOF"),
    (axes[1], temporal, "Leave-one-year-out"),
]:
    observed = table["Kc_target"].to_numpy(float)
    predicted = table["prediction"].to_numpy(float)
    metrics = regression_metrics(observed, predicted)
    lower, upper = identity_limits(observed, predicted)

    ax.scatter(observed, predicted, s=16, alpha=0.45)
    ax.plot([lower, upper], [lower, upper], linewidth=1.0)
    ax.set_xlim(lower, upper)
    ax.set_ylim(lower, upper)
    ax.set_xlabel("MODIS-derived Kc target")
    ax.set_ylabel("Ridge25 OOF Kc")
    ax.set_title(title)
    ax.text(
        0.04,
        0.96,
        (
            f"n = {metrics['n']}\n"
            f"R² = {metrics['R2']:.3f}\n"
            f"RMSE = {metrics['RMSE']:.3f}\n"
            f"MAE = {metrics['MAE']:.3f}\n"
            f"Bias = {metrics['BIAS']:.3f}"
        ),
        transform=ax.transAxes,
        va="top",
    )

protocols = ["Spatial OOF", "LOYO"]
r2_values = [
    metadata["spatial_metrics"]["R2"],
    metadata["temporal_metrics"]["R2"],
]
rmse_values = [
    metadata["spatial_metrics"]["RMSE"],
    metadata["temporal_metrics"]["RMSE"],
]

x = np.arange(2)
width = 0.35
axes[2].bar(x - width / 2, r2_values, width, label="R²")
axes[2].bar(x + width / 2, rmse_values, width, label="RMSE")
axes[2].axhline(0, linewidth=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(protocols)
axes[2].set_title("Protocol summary")
axes[2].legend(frameon=False)

fig.suptitle(
    "Final Ridge25 validation — MODIS-derived Kc target",
    fontsize=13,
)
fig.tight_layout()
save_figure(fig, "D01_validation_spatial_temporal.png")
plt.show()

validation

## D02 — Kc time series by station

The five panels show the MODIS-derived target together with spatial OOF and
LOYO predictions. This is useful for identifying phase, amplitude, seasonal
tracking and persistent site-specific bias rather than relying only on pooled metrics.

In [ ]:
series = spatial.merge(
    temporal[["station_id", "period_start", "prediction"]].rename(
        columns={"prediction": "prediction_loyo"}
    ),
    on=["station_id", "period_start"],
    how="left",
    validate="one_to_one",
).rename(columns={"prediction": "prediction_spatial"})

stations = ["ST01", "ST02", "ST03", "ST04", "ST05"]

fig, axes = plt.subplots(
    len(stations),
    1,
    figsize=(12.0, 12.0),
    sharex=True,
    constrained_layout=True,
)

for ax, station_id in zip(axes, stations):
    group = (
        series.loc[series["station_id"].astype(str).eq(station_id)]
        .sort_values("period_start")
        .copy()
    )

    ax.plot(
        group["period_start"],
        group["Kc_target"],
        linewidth=1.2,
        marker="o",
        markersize=2.5,
        label="MODIS-derived Kc target",
    )
    ax.plot(
        group["period_start"],
        group["prediction_spatial"],
        linewidth=1.0,
        label="Spatial OOF",
    )
    ax.plot(
        group["period_start"],
        group["prediction_loyo"],
        linewidth=1.0,
        label="LOYO",
    )
    ax.set_ylabel("Kc")
    ax.set_title(
        f"{station_id} — {STATION_NAMES.get(station_id, station_id)}",
        loc="left",
        fontsize=10,
    )

axes[0].legend(frameon=False, ncol=3)
axes[-1].set_xlabel("Period start")
fig.suptitle("Temporal behaviour of the final Kc model", fontsize=13)

save_figure(fig, "D02_station_kc_timeseries.png")
plt.show()

## D03 — Spatial transfer by held-out station

Each station corresponds to one study site and one land-cover context.
The panels therefore diagnose **site-specific spatial transfer**, not replicated
land-cover performance.

In [ ]:
station_metrics = pd.read_csv(STATION_METRICS)
station_metrics["station_id"] = station_metrics["station_id"].astype(str)
station_metrics = station_metrics.sort_values("station_id")

labels = [
    f"{sid}\n{STATION_NAMES.get(sid, sid)}"
    for sid in station_metrics["station_id"]
]

fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.2))
metrics_to_plot = [
    ("R2", "R²"),
    ("RMSE", "RMSE (Kc)"),
    ("MAE", "MAE (Kc)"),
    ("BIAS", "Bias (Kc)"),
]

for ax, (column, label) in zip(axes.flat, metrics_to_plot):
    ax.bar(labels, station_metrics[column])
    if column in {"R2", "BIAS"}:
        ax.axhline(0, linewidth=0.8)
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.tick_params(axis="x", labelrotation=20)

fig.suptitle("Spatial OOF transfer performance by station", fontsize=13)
fig.tight_layout()

save_figure(fig, "D03_spatial_transfer_by_station.png")
plt.show()

station_metrics

## D04 — Fitted Ridge structure and predictor redundancy

The old 30 m workflow compared permutation and Gini importance. That is not
appropriate here. The current model is Ridge, so this diagnostic shows:

1. standardized fitted coefficients; and
2. the strongest pairwise correlations in the actual training population.

Coefficient magnitude is **not** interpreted as causal importance, especially
under the strong optical collinearity visible in the second panel.

In [ ]:
parameters = pd.read_csv(MODEL_PARAMETERS)
training = pd.read_csv(TRAINING)

feature_names = parameters["feature"].tolist()
available_features = [f for f in feature_names if f in training.columns]

coef = (
    parameters
    .assign(abs_coefficient=lambda d: d["ridge_coefficient_standardized"].abs())
    .sort_values("abs_coefficient")
)

corr = training[available_features].corr()
pairs = []
for i, first in enumerate(available_features):
    for second in available_features[i + 1:]:
        value = corr.loc[first, second]
        if np.isfinite(value):
            pairs.append((first, second, value, abs(value)))

pair_table = (
    pd.DataFrame(
        pairs,
        columns=["feature_1", "feature_2", "r", "abs_r"],
    )
    .sort_values("abs_r", ascending=False)
    .head(15)
    .sort_values("abs_r")
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13.0, 7.5),
    gridspec_kw={"width_ratios": [1.1, 1.0]},
)

axes[0].barh(
    coef["feature"],
    coef["ridge_coefficient_standardized"],
)
axes[0].axvline(0, linewidth=0.8)
axes[0].set_xlabel("Standardized Ridge coefficient")
axes[0].set_title("(a) Final Ridge25 coefficient structure", loc="left")

pair_labels = [
    f"{a} × {b}"
    for a, b in zip(pair_table["feature_1"], pair_table["feature_2"])
]
axes[1].barh(pair_labels, pair_table["abs_r"])
axes[1].axvline(0.90, linewidth=0.8, linestyle="--")
axes[1].set_xlabel("|Pearson r|")
axes[1].set_title("(b) Strongest training-feature correlations", loc="left")

fig.suptitle(
    "Model structure must be interpreted together with predictor redundancy",
    fontsize=12,
)
fig.tight_layout()

save_figure(fig, "D04_ridge_structure_and_collinearity.png")
plt.show()

pair_table.sort_values("abs_r", ascending=False)

## D05 — Persistence baseline

Persistence is evaluated only on matched populations. This figure makes explicit
whether Ridge25 improves coarse-scale temporal prediction relative to carrying
forward the previous available Kc value.

In [ ]:
persistence = pd.read_csv(PERSISTENCE)

definition_order = [
    "previous_available_observation",
    "strict_previous_8day_composite",
    "previous_observation_within_16days",
]
definition_labels = {
    "previous_available_observation": "Previous\navailable",
    "strict_previous_8day_composite": "Strict previous\n8-day",
    "previous_observation_within_16days": "Previous within\n16 days",
}

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))

for ax, metric in zip(axes, ["R2", "RMSE", "MAE"]):
    x = np.arange(len(definition_order))
    width = 0.36

    persistence_values = []
    ridge_values = []

    for definition in definition_order:
        subset = persistence.loc[persistence["definition"].eq(definition)]
        persistence_values.append(
            float(subset.loc[subset["model"].eq("persistence"), metric].iloc[0])
        )
        ridge_values.append(
            float(
                subset.loc[
                    subset["model"].str.contains("Ridge25"),
                    metric,
                ].iloc[0]
            )
        )

    ax.bar(x - width / 2, persistence_values, width, label="Persistence")
    ax.bar(x + width / 2, ridge_values, width, label="Ridge25 spatial OOF")
    if metric == "R2":
        ax.axhline(0, linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [definition_labels[d] for d in definition_order],
        fontsize=8,
    )
    ax.set_ylabel(metric)
    ax.set_title(metric)

axes[0].legend(frameon=False)
fig.suptitle("Matched-population persistence baseline", fontsize=13)
fig.tight_layout()

save_figure(fig, "D05_persistence_baseline.png")
plt.show()

persistence

## D06 — Area of applicability

The current AOA is the accepted equal-weight standardized DI formulation.
This diagnostic does not claim that DI predicts model error; it explicitly tests
that relationship and shows the hard threshold used for publication.

In [ ]:
aoa = pd.read_csv(AOA_TRAINING, parse_dates=["period_start"])
aoa["station_id"] = aoa["station_id"].astype(str)

merged = spatial.merge(
    aoa,
    on=["station_id", "period_start", "spatial_block", "year"],
    how="left",
    validate="one_to_one",
)
merged["abs_error"] = merged["error"].abs()

threshold = float(aoa_metadata["threshold"])
inside = merged["training_di"] <= threshold

rho, rho_p = spearmanr(
    merged["training_di"],
    merged["abs_error"],
    nan_policy="omit",
)

rmse_inside = float(
    np.sqrt(np.mean(merged.loc[inside, "error"].to_numpy(float) ** 2))
)
rmse_outside = float(
    np.sqrt(np.mean(merged.loc[~inside, "error"].to_numpy(float) ** 2))
)

map_di = {}
for date in FINAL_DATES:
    raster_path = next(
        (WORKSPACE / "current" / "rasters" / date).glob("ET_ridge25_*_20m.tif")
    )
    with rasterio.open(raster_path) as src:
        descriptions = list(src.descriptions)
        di_index = descriptions.index("dissimilarity_index") + 1
        di = src.read(di_index, masked=True).compressed()
        if len(di) > 200_000:
            step = max(1, len(di) // 200_000)
            di = di[::step]
        map_di[date] = di

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0))

axes[0, 0].hist(aoa["training_di"], bins=45, alpha=0.75)
axes[0, 0].axvline(threshold, linestyle="--", linewidth=1.2)
axes[0, 0].set_xlabel("Training dissimilarity index")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].set_title(
    f"(a) Training DI and AOA threshold = {threshold:.3f}",
    loc="left",
)

for date in FINAL_DATES:
    axes[0, 1].hist(
        map_di[date],
        bins=60,
        density=True,
        histtype="step",
        linewidth=1.2,
        label=date,
    )
axes[0, 1].axvline(threshold, linestyle="--", linewidth=1.2)
axes[0, 1].set_xlabel("Raster dissimilarity index")
axes[0, 1].set_ylabel("Density")
axes[0, 1].set_title("(b) DI distributions in final map dates", loc="left")
axes[0, 1].legend(frameon=False)

axes[1, 0].scatter(
    merged["training_di"],
    merged["abs_error"],
    s=16,
    alpha=0.45,
)
axes[1, 0].axvline(threshold, linestyle="--", linewidth=1.2)
axes[1, 0].set_xlabel("Training DI")
axes[1, 0].set_ylabel("|Spatial OOF error| (Kc)")
axes[1, 0].set_title(
    f"(c) Error discrimination: Spearman ρ = {rho:.3f}",
    loc="left",
)

axes[1, 1].bar(
    ["Inside AOA", "Outside AOA"],
    [rmse_inside, rmse_outside],
)
axes[1, 1].set_ylabel("Spatial OOF RMSE (Kc)")
axes[1, 1].set_title(
    (
        "(d) OOF error by training-DI class\n"
        f"inside n={inside.sum()}, outside n={(~inside).sum()}"
    ),
    loc="left",
)

fig.suptitle(
    "Final equal-weight area-of-applicability diagnostic",
    fontsize=13,
)
fig.tight_layout()

save_figure(fig, "D06_area_of_applicability.png")
plt.show()

pd.DataFrame(
    {
        "threshold": [threshold],
        "spearman_rho_DI_abs_error": [rho],
        "spearman_p": [rho_p],
        "RMSE_inside": [rmse_inside],
        "RMSE_outside": [rmse_outside],
        "n_inside": [int(inside.sum())],
        "n_outside": [int((~inside).sum())],
    }
)

## D07 — Temporal ET comparison by station

This figure is the direct counterpart of the former five-station time-series diagnostic.

For each station, it compares:

- **Field-derived ET proxy**
- **MODIS parent ET**
- **Ridge25 downscaled ET**

Only periods available in the final field-comparison table are plotted.

Interpretation remains conservative:
- field ET is a proxy derived from reference ET and Kc;
- ST04 and ST05 use NDVI-derived Kc and therefore provide exploratory/sensitivity evidence;
- the comparison does not constitute independent validation of 20 m ET.

In [ ]:
field_time = pd.read_csv(FIELD_COMPARISON, parse_dates=["period_start"])
field_time["station_id"] = field_time["station_id"].astype(str)

required_columns = [
    "station_id",
    "period_start",
    "ET_field_proxy_mm_period",
    "ET_MODIS_parent_mm_period",
    "ET_Ridge_with_AOA_mm_period",
]
missing_columns = [c for c in required_columns if c not in field_time.columns]
if missing_columns:
    raise KeyError(
        "Missing required columns in field_comparison_scenarios.csv: "
        + ", ".join(missing_columns)
    )

station_order = ["ST01", "ST02", "ST03", "ST04", "ST05"]

fig, axes = plt.subplots(
    len(station_order),
    1,
    figsize=(12.0, 12.5),
    sharex=True,
    constrained_layout=True,
)

for ax, station_id in zip(axes, station_order):
    subset = (
        field_time.loc[field_time["station_id"].eq(station_id)]
        .dropna(
            subset=[
                "ET_field_proxy_mm_period",
                "ET_MODIS_parent_mm_period",
                "ET_Ridge_with_AOA_mm_period",
            ]
        )
        .sort_values("period_start")
        .copy()
    )

    ax.plot(
        subset["period_start"],
        subset["ET_field_proxy_mm_period"],
        marker="o",
        linewidth=1.4,
        markersize=4,
        label="Field-derived ET proxy",
    )
    ax.plot(
        subset["period_start"],
        subset["ET_MODIS_parent_mm_period"],
        marker="s",
        linewidth=1.2,
        markersize=3.5,
        label="MODIS parent ET",
    )
    ax.plot(
        subset["period_start"],
        subset["ET_Ridge_with_AOA_mm_period"],
        marker="^",
        linewidth=1.2,
        markersize=3.5,
        label="Ridge25 downscaled ET",
    )

    ax.set_ylabel("ET (mm / 8-day period)")
    ax.set_title(
        f"{station_id} — {STATION_NAMES.get(station_id, station_id)} "
        f"(n={len(subset)})",
        loc="left",
        fontsize=10,
    )
    ax.grid(False)

    if station_id in {"ST04", "ST05"}:
        ax.text(
            0.995,
            0.92,
            "NDVI-derived Kc proxy",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=7.5,
        )

axes[0].legend(
    frameon=False,
    ncol=3,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.30),
)
axes[-1].set_xlabel("Period start")

fig.suptitle(
    "Temporal comparison of field-derived ET proxy, MODIS parent ET "
    "and Ridge25 downscaled ET",
    fontsize=13,
)

save_figure(fig, "D07_station_ET_timeseries_field_modis_ridge.png")
plt.show()

## D08 — Field-derived ET proxy comparison

This is the current equivalent of the former "central test" figure.

Two distinctions are kept explicit:
- **all sites within AOA** includes ST04–ST05, whose Kc is NDVI-derived and is
  therefore exploratory/sensitivity evidence;
- **ST01–ST03 within AOA** uses the pre-specified literature Kc proxies.

The plots are comparisons with a field-derived ET proxy, not independent
validation of 20 m evapotranspiration.

In [ ]:
field = pd.read_csv(FIELD_COMPARISON, parse_dates=["period_start"])
field["station_id"] = field["station_id"].astype(str)
field_metrics = pd.read_csv(FIELD_METRICS)
field_sensitivity = pd.read_csv(FIELD_SENSITIVITY)

def scenario_subset(membership, ridge_column):
    subset = field.loc[field[membership].astype(bool)].dropna(
        subset=[
            "ET_field_proxy_mm_period",
            "ET_MODIS_parent_mm_period",
            ridge_column,
        ]
    ).copy()
    return subset

all_aoa = scenario_subset(
    "included_all_with_AOA",
    "ET_Ridge_with_AOA_mm_period",
)
fixed_aoa = scenario_subset(
    "included_fixed_Kc_with_AOA",
    "ET_Ridge_with_AOA_mm_period",
)

fig, axes = plt.subplots(2, 2, figsize=(11.5, 9.0))

for ax, subset, title in [
    (
        axes[0, 0],
        all_aoa,
        f"(a) All sites within AOA (n={len(all_aoa)})",
    ),
    (
        axes[0, 1],
        fixed_aoa,
        f"(b) ST01–ST03 fixed-Kc subset within AOA (n={len(fixed_aoa)})",
    ),
]:
    observed = subset["ET_field_proxy_mm_period"].to_numpy(float)
    modis = subset["ET_MODIS_parent_mm_period"].to_numpy(float)
    ridge = subset["ET_Ridge_with_AOA_mm_period"].to_numpy(float)

    lower, upper = identity_limits(observed, modis, ridge)

    ax.scatter(
        observed,
        modis,
        marker="o",
        s=42,
        alpha=0.7,
        label="MODIS parent",
    )
    ax.scatter(
        observed,
        ridge,
        marker="^",
        s=48,
        alpha=0.7,
        label="Ridge25 OOF",
    )
    ax.plot([lower, upper], [lower, upper], linewidth=1.0)
    ax.set_xlim(lower, upper)
    ax.set_ylim(lower, upper)
    ax.set_xlabel("Field-derived ET proxy (mm / 8-day period)")
    ax.set_ylabel("Satellite/model ET (mm / 8-day period)")
    ax.set_title(title, loc="left")

    for _, row in subset.iterrows():
        ax.annotate(
            row["station_id"],
            (
                row["ET_field_proxy_mm_period"],
                row["ET_Ridge_with_AOA_mm_period"],
            ),
            xytext=(2, 2),
            textcoords="offset points",
            fontsize=6,
            alpha=0.6,
        )

axes[0, 0].legend(frameon=False)

scenario_order = [
    "all_without_AOA_pure_extrapolations",
    "all_with_AOA",
    "fixed_Kc_without_AOA_pure_extrapolations",
    "fixed_Kc_with_AOA",
]
scenario_labels = [
    "All\nwithout AOA",
    "All\nwith AOA",
    "ST01–03\nwithout AOA",
    "ST01–03\nwith AOA",
]

x = np.arange(len(scenario_order))
width = 0.36
modis_rmse = []
ridge_rmse = []
n_values = []

for scenario in scenario_order:
    subset = field_metrics.loc[field_metrics["scenario"].eq(scenario)]
    modis_row = subset.loc[subset["model"].eq("MODIS_parent")].iloc[0]
    ridge_row = subset.loc[subset["model"].str.contains("Ridge25")].iloc[0]
    modis_rmse.append(float(modis_row["RMSE"]))
    ridge_rmse.append(float(ridge_row["RMSE"]))
    n_values.append(int(ridge_row["n"]))

axes[1, 0].bar(x - width / 2, modis_rmse, width, label="MODIS parent")
axes[1, 0].bar(x + width / 2, ridge_rmse, width, label="Ridge25 OOF")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(
    [f"{label}\n(n={n})" for label, n in zip(scenario_labels, n_values)],
    fontsize=8,
)
axes[1, 0].set_ylabel("RMSE (mm / 8-day period)")
axes[1, 0].set_title("(c) Four frozen field scenarios", loc="left")
axes[1, 0].legend(frameon=False)

sensitivity = field_sensitivity.loc[
    field_sensitivity["scenario"].eq("all_with_AOA")
].copy()

x2 = np.arange(2)
width2 = 0.36
completeness_order = ["primary_5of8_or_more", "complete_8of8_only"]
completeness_labels = ["≥5/8 days", "8/8 days"]

modis_values = []
ridge_values = []
sensitivity_n = []

for completeness in completeness_order:
    subset = sensitivity.loc[
        sensitivity["field_completeness"].eq(completeness)
    ]
    modis_row = subset.loc[subset["model"].eq("MODIS_parent")].iloc[0]
    ridge_row = subset.loc[subset["model"].str.contains("Ridge25")].iloc[0]
    modis_values.append(float(modis_row["RMSE"]))
    ridge_values.append(float(ridge_row["RMSE"]))
    sensitivity_n.append(int(ridge_row["n"]))

axes[1, 1].bar(x2 - width2 / 2, modis_values, width2, label="MODIS parent")
axes[1, 1].bar(x2 + width2 / 2, ridge_values, width2, label="Ridge25 OOF")
axes[1, 1].set_xticks(x2)
axes[1, 1].set_xticklabels(
    [
        f"{label}\n(n={n})"
        for label, n in zip(completeness_labels, sensitivity_n)
    ]
)
axes[1, 1].set_ylabel("RMSE (mm / 8-day period)")
axes[1, 1].set_title(
    "(d) Temporal-completeness sensitivity — all sites within AOA",
    loc="left",
)
axes[1, 1].legend(frameon=False)

fig.suptitle(
    "Field-derived ET proxy comparison — descriptive evidence only",
    fontsize=13,
)
fig.tight_layout()

save_figure(fig, "D08_field_proxy_comparison.png")
plt.show()

field_metrics

## D09 — Production support and reconciliation QA

The former 30 m workflow displayed a generic mass-conservation correction.
The current production contract is different: exact conservation applies to the
**full reconciled MODIS support before the publication mask**.

This figure therefore reports:
- published area by date;
- common spatial support;
- exact-overlap conservation error;
- support retained and negative values affected by the physical floor.

In [ ]:
raster_summary = pd.read_csv(RASTER_SUMMARY)

metadata_rows = []
for date in FINAL_DATES:
    folder = WORKSPACE / "current" / "rasters" / date
    metadata_path = next(folder.glob("production_metadata_*.json"))
    item = json.loads(metadata_path.read_text(encoding="utf-8"))
    metadata_rows.append(
        {
            "date": date,
            "eligible_modis_parents": item["eligible_modis_parents"],
            "publishable_active_fraction_of_active_support": item[
                "publishable_active_fraction_of_active_support"
            ],
            "negative_publishable_before_floor": item[
                "negative_publishable_before_floor"
            ],
            "max_error_before_floor": item[
                "max_abs_conservation_error_before_floor_mm"
            ],
            "max_error_after_floor": item[
                "max_abs_conservation_error_after_floor_mm"
            ],
            "tolerance": item["conservation_tolerance_mm"],
        }
    )

production = pd.DataFrame(metadata_rows)

published = (
    raster_summary.loc[raster_summary["scope"].eq("published")]
    .set_index("date")
    .loc[FINAL_DATES]
    .reset_index()
)
common = (
    raster_summary.loc[raster_summary["scope"].eq("common_all_dates")]
    .set_index("date")
    .loc[FINAL_DATES]
    .reset_index()
)

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0))

axes[0, 0].bar(published["date"], published["area_km2"])
axes[0, 0].set_ylabel("Published area (km²)")
axes[0, 0].set_title("(a) Published fine-resolution support", loc="left")
axes[0, 0].tick_params(axis="x", labelrotation=20)

axes[0, 1].bar(
    common["date"],
    100 * common["fraction_of_published_support"],
)
axes[0, 1].set_ylabel("Common support (% of published area)")
axes[0, 1].set_title(
    "(b) Support shared by all three map dates",
    loc="left",
)
axes[0, 1].tick_params(axis="x", labelrotation=20)

axes[1, 0].semilogy(
    production["date"],
    production["max_error_before_floor"],
    marker="o",
    label="Before floor",
)
axes[1, 0].semilogy(
    production["date"],
    production["max_error_after_floor"],
    marker="s",
    label="After floor",
)
axes[1, 0].axhline(
    production["tolerance"].iloc[0],
    linestyle="--",
    linewidth=1.0,
    label="Tolerance",
)
axes[1, 0].set_ylabel("Maximum conservation error (mm)")
axes[1, 0].set_title(
    "(c) Exact-overlap conservation on full reconciled support",
    loc="left",
)
axes[1, 0].tick_params(axis="x", labelrotation=20)
axes[1, 0].legend(frameon=False)

x = np.arange(len(FINAL_DATES))
axes[1, 1].bar(
    x - 0.18,
    100 * production["publishable_active_fraction_of_active_support"],
    width=0.36,
    label="Publishable fraction (%)",
)
ax2 = axes[1, 1].twinx()
ax2.bar(
    x + 0.18,
    production["negative_publishable_before_floor"],
    width=0.36,
    alpha=0.55,
    label="Negative cells before floor",
)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(FINAL_DATES, rotation=20)
axes[1, 1].set_ylabel("Publishable fraction of active support (%)")
ax2.set_ylabel("Negative cells before floor")
axes[1, 1].set_title(
    "(d) Publication support and physical-floor incidence",
    loc="left",
)

handles1, labels1 = axes[1, 1].get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
axes[1, 1].legend(
    handles1 + handles2,
    labels1 + labels2,
    frameon=False,
    loc="upper left",
)

fig.suptitle(
    "Final production QA — support, conservation and publication mask",
    fontsize=13,
)
fig.tight_layout()

save_figure(fig, "D09_production_support_and_conservation.png")
plt.show()

production

## Output inventory

These are local diagnostic figures. They do not alter or replace the frozen
scientific products.

In [ ]:
created = sorted(FIGURE_DIR.glob("D*.png"))
print(f"Created {len(created)} diagnostic figures")
for path in created:
    print(path.name)